In [169]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import pandas as pd
import numpy as np

import nltk
from nltk.tokenize import sent_tokenize, word_tokenize

In [170]:
print(f"Using the PyTorch version: {torch.__version__}")

## getting the device
device = "cuda" if torch.cuda.is_available() else 'cpu'
print(f"Device:{device}")

Using the PyTorch version: 2.8.0+cu128
Device:cuda


In [171]:
paragraph = """Donald Trump’s tariff policy became one of the most controversial features of modern U.S. economic strategy. A tariff is a tax placed on imported goods, and during Trump’s presidency and later trade actions associated with his agenda, tariffs were used as an instrument to protect domestic industries, reduce trade deficits, and pressure trading partners into changing their policies. China was the main target, but the effects extended to many countries and industries across the world. While supporters argued that tariffs would revive manufacturing and strengthen economic independence, critics pointed out that global trade is deeply interconnected, meaning that barriers imposed by one major economy can quickly affect producers, consumers, workers, and governments everywhere. In this way, Trump’s tariffs became not just a national trade policy but a global economic issue.
One of the clearest impacts of Trump’s tariffs was the disruption of international trade patterns. When the United States raised tariffs on imported goods, especially from China, foreign products became more expensive in the American market. Importers, retailers, and manufacturers had to either absorb these costs or pass them on to buyers. Many firms responded by reducing orders from tariff-hit countries and moving sourcing to other nations such as Vietnam, Mexico, and India. This process, often described as trade diversion, changed global supply chains but did not necessarily make production more efficient. Instead, companies spent time and money restructuring their operations, locating alternative suppliers, and managing customs complexity. These adjustments created delays and increased uncertainty, especially for multinational firms that depended on predictable cross-border trade.
Another major consequence was the increase in production costs. Modern manufacturing depends on global supply networks in which raw materials, components, and finished products cross borders multiple times. Tariffs on steel, aluminum, electronics, machinery, and intermediate goods raised costs for industries that relied on imported inputs. As a result, even some U.S. companies that tariffs were supposed to protect ended up paying more for materials and parts. Higher costs reduced competitiveness for exporters and lowered profit margins for domestic producers. In many cases, businesses responded by increasing prices, cutting investment, delaying expansion, or reducing hiring. This showed that tariffs do not affect only foreign producers; they often create pressure throughout the domestic economy as well.
Consumers also felt the effects. Since tariffs function like a tax on imports, part of the burden often falls on households through higher prices for everyday goods. Items such as electronics, appliances, industrial products, and consumer goods became more expensive when companies passed on the additional import cost. Even when price increases were not dramatic for each single item, the overall inflationary effect could be significant across the economy. At a time when many countries were already dealing with fragile recovery, interest rate pressures, and market uncertainty, tariff-driven price increases added another layer of stress. This weakened consumer purchasing power and made economic conditions more difficult, especially for low- and middle-income families who are more sensitive to price changes.
Trump’s tariffs also triggered retaliation from other countries, which intensified the damage. When a major economy imposes tariffs, affected countries often respond by placing tariffs on exports from that country. This happened in sectors such as agriculture, manufacturing, and industrial trade. American farmers, for example, were among those hurt when foreign governments targeted agricultural exports in response to U.S. tariff measures. Retaliatory action reduced export opportunities, disrupted established markets, and forced governments to consider support programs for affected industries. More broadly, this cycle of action and reaction reduced trust between trading partners and increased the risk of prolonged trade wars. Once retaliation begins, it becomes harder to restore normal trade relations because political and economic tensions become closely linked.
The global economy was affected not only through direct trade losses but also through uncertainty. Businesses prefer stable rules, predictable costs, and reliable access to markets. Trump’s tariff policies created uncertainty about future trade conditions, future regulations, and the possibility of further restrictions. Investors became more cautious, companies delayed long-term decisions, and financial markets reacted nervously to trade announcements and negotiations. This uncertainty can be just as harmful as the tariffs themselves because it reduces confidence, weakens capital spending, and slows growth. Research from major institutions suggested that large-scale tariff measures and the fear of escalation could lower both U.S. and global GDP growth. The negative effect was especially strong in export-dependent economies and in countries tied closely to global manufacturing chains.
At the international level, Trump’s tariffs also challenged the rules-based global trading order. For decades, global trade has been shaped by agreements, institutions, and negotiated dispute mechanisms designed to reduce arbitrary barriers and encourage cooperation. Aggressive tariff use by the United States signaled a shift toward unilateral economic nationalism, where national advantage is prioritized over multilateral coordination. This weakened confidence in institutions such as the World Trade Organization and encouraged other countries to consider more defensive or protectionist policies of their own. As more nations react by safeguarding domestic industries, the result can be a fragmented global economy with more barriers, less efficiency, and slower overall growth.
At the same time, some countries found limited opportunities in the changing trade environment. As supply chains moved away from China in response to tariffs, alternative manufacturing hubs such as Vietnam, Mexico, and India attracted some investment and export business. However, these gains were uneven and did not fully offset the broader costs to global trade. Relocating production is expensive, and new supply chains take time to build. In addition, when trade policy becomes unpredictable, companies may hesitate to commit fully to new markets. Therefore, while some economies benefited from trade diversion, the overall effect on the world economy remained mixed and unstable rather than clearly positive.
In conclusion, Trump’s tariffs had a far-reaching impact on global trade and the economy. They raised costs for businesses, increased prices for consumers, disrupted supply chains, provoked retaliation, and weakened international economic confidence. Although the policy aimed to protect domestic industry and create strategic leverage, its wider consequences showed how interconnected the modern global economy has become. Rather than producing simple national gains, the tariffs contributed to slower growth, greater uncertainty, and a more fragmented trading system. In the long run, the experience demonstrated that trade policy can shape not only markets and industries but also the stability of the global economic order itself.
"""

In [172]:
# Create a vocab
vocab = {'<UNK>':0}

# Tokenizing the word
tokens = word_tokenize(paragraph.lower())
tokens

['donald',
 'trump',
 '’',
 's',
 'tariff',
 'policy',
 'became',
 'one',
 'of',
 'the',
 'most',
 'controversial',
 'features',
 'of',
 'modern',
 'u.s.',
 'economic',
 'strategy',
 '.',
 'a',
 'tariff',
 'is',
 'a',
 'tax',
 'placed',
 'on',
 'imported',
 'goods',
 ',',
 'and',
 'during',
 'trump',
 '’',
 's',
 'presidency',
 'and',
 'later',
 'trade',
 'actions',
 'associated',
 'with',
 'his',
 'agenda',
 ',',
 'tariffs',
 'were',
 'used',
 'as',
 'an',
 'instrument',
 'to',
 'protect',
 'domestic',
 'industries',
 ',',
 'reduce',
 'trade',
 'deficits',
 ',',
 'and',
 'pressure',
 'trading',
 'partners',
 'into',
 'changing',
 'their',
 'policies',
 '.',
 'china',
 'was',
 'the',
 'main',
 'target',
 ',',
 'but',
 'the',
 'effects',
 'extended',
 'to',
 'many',
 'countries',
 'and',
 'industries',
 'across',
 'the',
 'world',
 '.',
 'while',
 'supporters',
 'argued',
 'that',
 'tariffs',
 'would',
 'revive',
 'manufacturing',
 'and',
 'strengthen',
 'economic',
 'independence',
 ',

In [173]:
for word in tokens:
    if word not in vocab:
        vocab[word] = len(vocab)

print(f"Vocabulary size: {len(vocab)}")
print(f"Sample vocabulary: {list(vocab.items())[:10]}")

Vocabulary size: 489
Sample vocabulary: [('<UNK>', 0), ('donald', 1), ('trump', 2), ('’', 3), ('s', 4), ('tariff', 5), ('policy', 6), ('became', 7), ('one', 8), ('of', 9)]


In [174]:
sentences = sent_tokenize(paragraph.lower())
sentences

['donald trump’s tariff policy became one of the most controversial features of modern u.s. economic strategy.',
 'a tariff is a tax placed on imported goods, and during trump’s presidency and later trade actions associated with his agenda, tariffs were used as an instrument to protect domestic industries, reduce trade deficits, and pressure trading partners into changing their policies.',
 'china was the main target, but the effects extended to many countries and industries across the world.',
 'while supporters argued that tariffs would revive manufacturing and strengthen economic independence, critics pointed out that global trade is deeply interconnected, meaning that barriers imposed by one major economy can quickly affect producers, consumers, workers, and governments everywhere.',
 'in this way, trump’s tariffs became not just a national trade policy but a global economic issue.',
 'one of the clearest impacts of trump’s tariffs was the disruption of international trade patterns

In [175]:
for sentence in sentences:
    print(f"Sentence: {sentence}")
    print(f"Tokenized Sentence: {word_tokenize(sentence)}")
    print('-'*20)

Sentence: donald trump’s tariff policy became one of the most controversial features of modern u.s. economic strategy.
Tokenized Sentence: ['donald', 'trump', '’', 's', 'tariff', 'policy', 'became', 'one', 'of', 'the', 'most', 'controversial', 'features', 'of', 'modern', 'u.s.', 'economic', 'strategy', '.']
--------------------
Sentence: a tariff is a tax placed on imported goods, and during trump’s presidency and later trade actions associated with his agenda, tariffs were used as an instrument to protect domestic industries, reduce trade deficits, and pressure trading partners into changing their policies.
Tokenized Sentence: ['a', 'tariff', 'is', 'a', 'tax', 'placed', 'on', 'imported', 'goods', ',', 'and', 'during', 'trump', '’', 's', 'presidency', 'and', 'later', 'trade', 'actions', 'associated', 'with', 'his', 'agenda', ',', 'tariffs', 'were', 'used', 'as', 'an', 'instrument', 'to', 'protect', 'domestic', 'industries', ',', 'reduce', 'trade', 'deficits', ',', 'and', 'pressure', 't

In [176]:
# defining a function to get indices

def get_index(vocab, sentence):

    indices = []

    for word in word_tokenize(sentence.lower()):
        if word in vocab:
            indices.append(vocab[word])
        else:
            indices.append(vocab['<UNK>'])
    
    return indices

# empty list to get array of sentences
input_sentences_arr = []

# create a loop to get sentence array
for sentence in sentences:
    input_sentences_arr.append(get_index(vocab, sentence))

print(f"Total number of sentences: {len(input_sentences_arr)}")

Total number of sentences: 55


In [177]:
# empty list for sequence array
sequential_arr = []

# lambda function to get the sequence of sentences
l_func = lambda x: [x[:i+1] for i in range(1, len(x))]


for sentence in input_sentences_arr:
    sequential_arr.extend(l_func(sentence))

print(f"Few Sequential Arrays in the list: \n{sequential_arr[:19]}")

Few Sequential Arrays in the list: 
[[1, 2], [1, 2, 3], [1, 2, 3, 4], [1, 2, 3, 4, 5], [1, 2, 3, 4, 5, 6], [1, 2, 3, 4, 5, 6, 7], [1, 2, 3, 4, 5, 6, 7, 8], [1, 2, 3, 4, 5, 6, 7, 8, 9], [1, 2, 3, 4, 5, 6, 7, 8, 9, 10], [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11], [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12], [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13], [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 9], [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 9, 14], [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 9, 14, 15], [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 9, 14, 15, 16], [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 9, 14, 15, 16, 17], [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 9, 14, 15, 16, 17, 18], [19, 5]]


In [178]:
max_count = max([len(sentence) for sentence in sequential_arr])
print(f"Maximum sequence length: {max_count}")

Maximum sequence length: 49


In [179]:
# Pre-Padding on the sequential array

padded_sequential_arr = []

for i in range(len(sequential_arr)):
    padded_sequential_arr.append(np.zeros(max_count - len(sequential_arr[i]), dtype=int).tolist() + sequential_arr[i])

In [180]:
padded_sequential_tensor = torch.tensor(padded_sequential_arr,dtype=torch.long)
padded_sequential_tensor.shape

torch.Size([1154, 49])

In [181]:
## Splitting the Data in Feature and Target

X = padded_sequential_tensor[:,:-1]
y = padded_sequential_tensor[:,-1]

In [182]:
class CustomDataset(Dataset):

    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, index):
        return self.X[index], self.y[index]

In [183]:
data = CustomDataset(X,y)

data[:5]

(tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
          0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
         [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
          0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 2],
         [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
          0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 2, 3],
         [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
          0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 2, 3, 4],
         [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
          0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 2, 3, 4, 5]]),
 tensor([2, 3, 4, 5, 6]))

In [184]:
dataloader = DataLoader(data, batch_size=32, shuffle=True)

In [185]:
# Defining the Vocabulary Size and the Model
vocab_size = len(vocab)

class NextWordPredictor(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()

        # Embedding layer --> (vocab_size, embedding_dim) embedding_dim --> 100, 
        # if we have 200 words in the vocab, we will represent each word as a 100-dimensional vector
        self.embeddings = nn.Embedding(vocab_size, 100) 
        self.lstm = nn.LSTM(100, 150, batch_first=True) # LSTM --> output, (hidden_state, cell_state)
        self.fc = nn.Linear(150, vocab_size)

    def forward(self, x):
        embedding_arr = self.embeddings(x)
        intermediate_hidden_state, (hidden_state, cell_state) = self.lstm(embedding_arr)
        output = self.fc(hidden_state.squeeze(0))

        return output

**NOTE:**

- `LSTM` already has activations internally.
- Adding an activation on the final `Linear` when using `CrossEntropyLoss` isn't required.
- As `nn.CrossEntropyLoss()` expects raw logits.
- It combines `LogSoftmax` and `NLLLoss` internally.
- We can use an activation earlier only if you add extra hidden layers, but it is not required for this simple next-word prediction model.

In [186]:
epochs = 128
learning_rate = 0.001

model = NextWordPredictor(vocab_size).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

In [187]:
for epoch in range(epochs):
    total_loss = 0

    for batch_X, batch_y in dataloader:

        batch_X, batch_y = batch_X.to(device), batch_y.to(device)

        optimizer.zero_grad()

        y_pred = model(batch_X)

        loss = criterion(y_pred, batch_y)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(dataloader)
    print(f"Epoch: {epoch+1:<3}/{epochs} | Loss: {total_loss:.10f} | Average Loss: {avg_loss:.10f}")

Epoch: 1  /128 | Loss: 225.6183514595 | Average Loss: 6.0977932827
Epoch: 2  /128 | Loss: 200.2359385490 | Average Loss: 5.4117821229
Epoch: 3  /128 | Loss: 186.7833733559 | Average Loss: 5.0481992799
Epoch: 4  /128 | Loss: 171.1692318916 | Average Loss: 4.6261954565
Epoch: 5  /128 | Loss: 160.7709403038 | Average Loss: 4.3451605488
Epoch: 6  /128 | Loss: 147.5223288536 | Average Loss: 3.9870899690
Epoch: 7  /128 | Loss: 132.6655108929 | Average Loss: 3.5855543485
Epoch: 8  /128 | Loss: 119.6842751503 | Average Loss: 3.2347101392
Epoch: 9  /128 | Loss: 109.1378171444 | Average Loss: 2.9496707336
Epoch: 10 /128 | Loss: 94.2477284074 | Average Loss: 2.5472359029
Epoch: 11 /128 | Loss: 86.0760462284 | Average Loss: 2.3263796278
Epoch: 12 /128 | Loss: 74.5687794685 | Average Loss: 2.0153724181
Epoch: 13 /128 | Loss: 64.3272541761 | Average Loss: 1.7385744372
Epoch: 14 /128 | Loss: 55.4295089245 | Average Loss: 1.4980948358
Epoch: 15 /128 | Loss: 48.3451422453 | Average Loss: 1.3066254661
E

In [190]:
def predict_next_word(model, vocab, text, max_count=max_count):

    # print(f"Input Sentence: {text}")
    ## Covert to indices
    sequence_arr = get_index(sentence=text, vocab=vocab)
    # print(f"Sentence Sequence List: {sequence_arr}")

    ## Padding
    padded_tensor = torch.tensor(np.zeros(max_count-len(sequence_arr), dtype=int).tolist() + sequence_arr, dtype=torch.long).unsqueeze(0).to(device)

    # print(f"Padded Tensor Shape: \n{padded_tensor}")

    predicted_word = model(padded_tensor)
    # print(f"Predicted Word Tensor: \n{predicted_word.shape}")

    val, idx = torch.max(predicted_word, dim=1)
    # print(f"Predicted Word Value: {val.item()} Index: {idx.item()}")
    # print(f"Predicted Word: {list(vocab.keys())[idx.item()]}")

    next_word = list(vocab.keys())[idx.item()]

    punctuation = ['.', ',', '!', '?', ';', ':', '-', '(', ')', '[', ']', '{', '}', '"', "'", "’", "s"]

    if next_word not in punctuation:
        return text + " " + next_word
    else:
        return text + next_word

predict_next_word(model, vocab, text="A tariff is a tax")

'A tariff is a tax placed'

In [192]:
number_of_predictions = 44
input_text = "A tariff is a tax"

for i in range(number_of_predictions):
    
    output = predict_next_word(model, vocab, text=input_text)
    print(f"{output}")
    input_text = output

A tariff is a tax placed
A tariff is a tax placed on
A tariff is a tax placed on imported
A tariff is a tax placed on imported goods
A tariff is a tax placed on imported goods,
A tariff is a tax placed on imported goods, and
A tariff is a tax placed on imported goods, and during
A tariff is a tax placed on imported goods, and during trump
A tariff is a tax placed on imported goods, and during trump’
A tariff is a tax placed on imported goods, and during trump’s
A tariff is a tax placed on imported goods, and during trump’s presidency
A tariff is a tax placed on imported goods, and during trump’s presidency and
A tariff is a tax placed on imported goods, and during trump’s presidency and later
A tariff is a tax placed on imported goods, and during trump’s presidency and later trade
A tariff is a tax placed on imported goods, and during trump’s presidency and later trade actions
A tariff is a tax placed on imported goods, and during trump’s presidency and later trade actions associated
A

### Notebook Summary

#### 1. Preprocessing
- The notebook tokenizes text using `word_tokenize` from NLTK.
- A vocabulary is built from the paragraph, starting with `{'<UNK>': 0}`.
- Each sentence is converted to a sequence of integer indices via `get_index()`.
- Sequential training examples are generated by creating incremental sub-sequences from each sentence.
- Sequences are pre-padded to the maximum length using zeros, then split into:
  - `X` = all tokens except the last
  - `y` = the next word target token

#### 2. Model Structure
- `NextWordPredictor` is a PyTorch model with:
  - `nn.Embedding(vocab_size, 100)`
  - `nn.LSTM(100, 150, batch_first=True)`
  - `nn.Linear(150, vocab_size)`
- In `forward()`, the model:
  - embeds the input sequence
  - passes embeddings through the LSTM
  - uses the final hidden state to predict logits over the entire vocabulary

#### 3. Metrics
- The notebook trains using:
  - `nn.CrossEntropyLoss()`
  - `optim.Adam(...)`
- During training, it prints epoch loss and average loss.
- No explicit accuracy metric is computed in the code.

#### 4. Observations
- The model predicts the next word for the current sentence context correctly with 100% accuracy.
- However, it fails to predict the next word from the upcoming sentence, because that target word was not preprocessed earlier as a training target.

#### 5. Improvement Suggestions
- Use a proper train/validation split so performance is measured more realistically.
- Add an accuracy metric or top-k accuracy during training.
- Expand preprocessing so the model can learn across sentence boundaries if you want next-sentence prediction.
- Consider using dropout, masking, or a larger dataset to reduce overfitting and improve generalization.